#  5. 動畫作品辨識的四種實現路線

此部分為開始將「前幾步的理論，開始實踐的路線藍圖」 

## 1. 問題定義

本專案希望建立一套動畫畫面辨識系統。  

使用者輸入一張動畫截圖，系統嘗試判斷該圖片屬於哪一部動畫作品。  

本專案將使用相同的動畫資料集與測試集，分別實作四種方法，觀察它們在辨識原理、準確率、速度、資料需求與可解釋性上的差異。  

## 數學的詮釋：  
一張輸入圖片(x是圖片，X是作品)：  $x∈X$     
動畫作品類別：$y∈{1,2,…,C}$   
最終希望得到：  $f(x)=y$  

所謂的預測模型就是期望 建立一個函數$f$，可以 $f(x)$ 可以直接對應到此圖片 $x$ 的作品 $y$



# 2. 為什麼選擇四條路線？

不是單純四種模型，而是四種不同的視覺思想：  

A. 圖片是否與曾見過的圖片相似？  
B. 圖片具有哪個動畫類別的特徵？  
C. 能否利用既有視覺知識學習動畫分類？  
D. 圖片在視覺特徵空間中接近哪些圖片？  

哪四種將會在下面詳細說明。

# 3. 路線 A：感知雜湊

### 3.1 核心思想

有沒有看過一張和它幾乎一樣的圖片？  
它不是真的理解角色、背景或畫風，而是將圖片壓縮成一串簡短的「視覺指紋」(其實就是一連串的數據，也可以說是個高維向量)

### 3.2 圖片如何轉換成雜湊值：介紹感知雜湊(Perceptual)
感知雜湊（Perceptual Hash, 簡稱 pHash）與傳統的密碼學雜湊（如 MD5, SHA-256）完全不同。密碼學雜湊具有「雪崩效應」，圖片只要改動 1 個像素，產生的雜湊值就會天差地遠；而感知雜湊則是為了「特徵提取」而設計，只要兩張圖片肉眼看起來相似（即使經過微調大小、壓縮、輕微色差），生成的雜湊值就會極為接近。  

將一張圖片轉換為感知雜湊值，核心邏輯是將高維度的圖像資訊，透過降維與頻域轉換，濃縮成一串代表結構特徵的位元列（Bitstring）。

最經典且效果優異的演算法為 pHash（Based on DCT），其數學與計算轉換過程主要分為 5 個步驟：

1. 縮放與灰階化（Preprocessing & Dimension Reduction）等比例縮放：將圖片強制縮放到 $32 \times 32$ 像素。此動作拋棄了圖片解析度與長寬比的高頻細節，只保留整體結構。灰階化：將 RGB 三通道轉換為單一灰階值（通常使用 $Y = 0.299R + 0.587G + 0.114B$），將資訊維度從 $32 \times 32 \times 3$ 降至 $32 \times 32$ 的純亮度矩陣 $A$。

2. 離散餘弦轉換（Discrete Cosine Transform, DCT）
DCT 是將圖像從「空間域（Spatial Domain）」轉換到「頻率域（Frequency Domain）」的關鍵步驟。
對於 $32 \times 32$ 的灰階矩陣 $A$，計算其雙變數 DCT 矩陣 $F$：$$F(u, v) = \frac{1}{4} C(u) C(v) \sum_{x=0}^{31} \sum_{y=0}^{31} A(x, y) \cos\left[\frac{(2x+1)u\pi}{64}\right] \cos\left[\frac{(2y+1)v\pi}{64}\right]$$  

頻域特性：轉換後的 $F(u, v)$ 矩陣中，左上角代表低頻成分（圖像的大輪廓、整體光影結構），右下角代表高頻成分（細節、噪訊、紋理）。

3. 擷取低頻矩陣（Low-Frequency Extraction）  
圖像的主要視覺特徵全部集中在低頻區。因此，直接捨棄右下角的高頻細節，僅截取左上角的 $8 \times 8$ 矩陣 $B$（共 64 個係數）。  
這一步驟實現了極高比例的資訊壓縮，同時保證了對縮放、壓縮與微小噪訊的抗性（Robustness）。  


4. 計算均值與二值化（Quantization & Binarization）  
去除 DC 分量：$B(0,0)$ 是直流分量（代表整張圖的平均亮度），通常予以排除或單獨處理，避免受到全局亮度的干擾。  
計算門檻值：計算剩餘 63 個（或包含 DC 的 64 個）低頻 DCT 係數的平均值 $\mu$：$$\mu = \frac{1}{N} \sum_{(u,v) \in \text{LowFreq}} B(u, v)$$  

產生位元：將每個係數與 $\mu$ 進行比較，建立二元邏輯：$$b_{i} = \begin{cases} 1, & \text{if } B(u, v) \ge \mu \\ 0, & \text{if } B(u, v) < \mu \end{cases}$$

5. 構建 64-bit 雜湊值（Hash Generation）  
將生成的 64 個 0 與 1 組合成一個 64 位的二進位字串，通常會轉寫為 16 進位表示（例如：a3f01c89e2b45f01）。這串數值即為該圖片的圖片指紋（Image Fingerprint）。

### 3.3 Hamming Distance  
得到兩張圖片的感知雜湊值後，比對兩圖是否相似不需要重新對比圖像本身，只需計算這兩個 64-bit 字串的漢明距離（即進行 XOR 運算後計算 1 的個數）：
$$D_H(H_1, H_2) = \text{popcount}(H_1 \oplus H_2)$$

$D_H = 0$：代表兩圖特徵高度一致或完全相同。  
$D_H \le 5$：通常判定為極度相似或同源圖片（僅有壓縮或微小裁切）。  
$D_H > 10$：代表兩圖視覺結構差異顯著，屬於不同圖片。  

### 3.4 適合解決的問題

擅長查找相似度高的圖片。  


### 3.5 優點與限制
如果使用者上傳的是同一部動畫中完全不同的場景，例如以下：  
A：芙莉蓮在森林中    
B：芙莉蓮在魔法考試會場  
雖然都是同一部動畫《葬送的芙莉蓮》，但整體畫面完全不同，圖片指紋也會相差很遠，在這個系統中就會判斷錯誤。  

所以它比較像「查原圖工具」，而不是：「理解作品風格的模型」

### 3.6 在本專案中的用途


### 核心流程：
圖片  
→ 縮小與簡化  
→ 產生圖片指紋  
→ 比較 Hamming Distance  
→ 找到最相似圖片  

# 4. 路線 B：從零訓練小型 CNN

### 4.1 核心思想

從大量標記好的圖片中，自己學習每部動畫的視覺規律。

CNN 一開始並不知道什麼是芙莉蓮，也不知道什麼是動畫。它的卷積核最初通常只是隨機數。
經過訓練後，它會逐步調整卷積核：

$$K_{\text{new}} = K_{\text{old}} - \eta \frac{\partial L}{\partial K}$$

其中：  
K：卷積核參數  
L：模型預測錯誤程度  
η：學習率  

也就是：哪些卷積核讓分類錯誤下降，就朝那個方向修改。

### 4.2 從人工 Kernel 到可學習 Kernel
大致上可能形成以下層次：

#### 前面幾層
學習較基礎的特徵：
1. 水平與垂直邊緣
2. 顏色變化
3. 線條粗細
4. 局部紋理
5. 明暗對比

這一部分和我之前做的 Sobel、Sharpen、Feature Map 有直接關係。

差別是：
以前：我親自指定 Kernel
現在：模型自行學習 Kernel

#### 中間幾層
可能開始組合成：  
1. 眼睛形狀
2. 頭髮輪廓
3. 服裝紋理
4. 臉部比例
5. 背景物件
6. 特效

#### 後面幾層
可能學習更整體的作品特徵：
1. 角色設計
2. 色彩風格
3. 場景構圖
4. 背景美術
5. 特定人物
6. 特定世界觀元素

### 4.3 卷積層、激活函數與池化層
### 4.4 分類層與類別機率
### 4.5 Loss 與反向傳播
### 4.6 優點與限制
即使測試圖片不是訓練資料中的原圖，只要視覺特徵相似，它仍可能辨識成功。

例如：
訓練資料：芙莉蓮在村莊、森林、城鎮
測試資料：芙莉蓮在洞窟

CNN 可能透過角色外觀、畫風、色彩或其他線索判斷作品。

### 4.7 在本專案中的用途

# 5. 路線 C：使用預訓練模型進行遷移學習

### 5.1 核心思想

先使用一個已經看過大量圖片的模型，再教它分辨你的幾部動畫。

路線 B 的 CNN 像是一個剛出生的學生，從0開始訓練。  
而路線 C 則是「像一個已經學過通用視覺知識的學生」，只是它還不知道你的任務中有哪些動畫。


### 5.2 什麼是預訓練模型

### 5.3 什麼是 Backbone

### 5.4 凍結特徵提取器

### 5.5 Fine-tuning

### 5.6 與 Custom CNN 的差異

### 5.7 優點與限制
優點
1. 資料量有限的分類問題  
2. 快速建立效果較好的模型  
3. 實際工程應用  
4. 作為 Custom CNN 的強力對照組  

### 5.8 在本專案中的用途